# AISEHack 2.0 — Polymer Property Prediction

Predicts two polymer properties (`tg`, `egc`) from SMILES strings using RDKit
molecular descriptors. Two modeling phases are compared honestly via 5-fold
CV, and the better model per target is used for the final submission.

**Phase 1:** Ridge regression on RDKit descriptors (baseline approach)
**Phase 2:** HistGradientBoosting on the same features (the improvement)

Two real bugs were found and fixed while building this — both documented
below, since they silently produce bad predictions if left in.

In [1]:
import pandas as pd
import numpy as np
import os
import json

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

## 1. Load data

Tries the Kaggle competition input path first, then falls back to a local
copy. If you're running this on Kaggle and it can't find the files, attach
the competition dataset to the notebook (Add Data) and update the slug
below to match whatever folder name Kaggle mounts it under.

In [2]:
def find_file(filename, slug_guesses=("aisehack-2-0",)):
    candidates = [f"/kaggle/input/{slug}/{filename}" for slug in slug_guesses]
    candidates += [f"/kaggle/input/{filename}", filename]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Couldn't find {filename}. Tried: {candidates}")

train = pd.read_csv(find_file("train.csv"))
test = pd.read_csv(find_file("test.csv"))

train_tg = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc = test[test['target_type'] == 'egc'].reset_index(drop=True)

print(f"train: tg={len(train_tg)}, egc={len(train_egc)}")
print(f"test:  tg={len(test_tg)}, egc={len(test_egc)}")

train: tg=4143, egc=2028
test:  tg=2763, egc=1352


## 2. RDKit descriptor calculation

Each polymer SMILES (using `*` to mark the repeat-unit attachment point) is
converted into RDKit's full default descriptor set (~217 physicochemical
descriptors: molecular weight, LogP, ring counts, topological indices, etc).
This takes a couple of minutes for ~10k molecules.

In [3]:
def calculate_descriptors(data):
    rows = []
    for smile in data['smiles']:
        mol = Chem.MolFromSmiles(smile)
        rows.append(Descriptors.CalcMolDescriptors(mol))
    return pd.DataFrame(rows)

tg_feat_data = calculate_descriptors(train_tg)
egc_feat_data = calculate_descriptors(train_egc)
tg_feat_test = calculate_descriptors(test_tg)
egc_feat_test = calculate_descriptors(test_egc)

tg_feat_data['target'] = train_tg['target'].values
egc_feat_data['target'] = train_egc['target'].values

X_tg_raw, y_tg = tg_feat_data.drop(columns=['target']), tg_feat_data['target']
X_egc_raw, y_egc = egc_feat_data.drop(columns=['target']), egc_feat_data['target']

## 3. Cleaning — two real bugs found here

A plain `fillna(mean())` on the raw descriptors is not enough:

1. **8 `BCUT2D_*` columns are `NaN` for every single row.** Every polymer
   SMILES here contains a `*` dummy attachment atom, which these particular
   descriptors can't handle — they fail for 100% of molecules, not just a
   few. `fillna(mean())` of an all-`NaN` column is still `NaN`, so these
   columns have to be dropped outright, not imputed.
2. **`Ipc` (an information-content descriptor) explodes to ~10⁵⁰–10¹⁰⁰⁺ for
   certain molecules** — a known RDKit quirk. Left in, it wrecks
   `StandardScaler` + `Ridge` numerically: the scaled design matrix's
   condition number blows past 1e50, and Ridge regression silently returns
   garbage coefficients (astronomical, meaningless RMSE) even with
   regularization. Dropping this one column fixes it completely.

The **test set** also needs the identical `inf`/`NaN` cleanup before
scoring — the most common way this kind of pipeline silently breaks is
cleaning train features carefully and then forgetting to do the same for
test, which produces `NaN` predictions Kaggle will reject or zero-score.

In [4]:
def clean_train_features(X):
    X = X.replace([np.inf, -np.inf], np.nan)
    drop_cols = X.columns[X.isnull().all()].tolist()
    if 'Ipc' in X.columns:
        drop_cols = drop_cols + ['Ipc']
    X = X.drop(columns=drop_cols)
    X = X.fillna(X.mean())
    return X, drop_cols

X_tg, dropped_tg = clean_train_features(X_tg_raw)
X_egc, dropped_egc = clean_train_features(X_egc_raw)
print("Dropped columns (tg): ", dropped_tg)
print("Dropped columns (egc):", dropped_egc)

Dropped columns (tg):  ['BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'Ipc']
Dropped columns (egc): ['BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'Ipc']


## 4. Phase 1 — Ridge regression baseline

5-fold CV picks the regularization strength (`alpha`) per target. The
search grid is narrowed to `1e-3 .. 1e2` — the original wider grid
(`1e-10 .. 1e1`) includes near-zero alpha values that are numerically
unstable given how collinear the RDKit descriptors are (15+ column pairs
correlate above 0.999), independent of the `Ipc` issue above.

We also check: does picking the "top 20 features by `|Ridge coefficient|`"
actually help? With this much collinearity, coefficient magnitude doesn't
reliably identify the most *independent* predictive features — credit gets
split arbitrarily across near-duplicate columns. Worth checking rather
than assuming.

In [5]:
def run_cv_ridge(X, y, alpha):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    errors = []
    for tr, va in kf.split(X):
        X_tr, X_va = X.values[tr], X.values[va]
        y_tr, y_va = y.values[tr], y.values[va]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_va_s = scaler.transform(X_va)
        model = Ridge(alpha).fit(X_tr_s, y_tr)
        errors.append(root_mean_squared_error(y_va, model.predict(X_va_s)))
    return np.mean(errors)

def cv_lr(X, y):
    alphas = np.logspace(-3, 2, 20)
    results = sorted(((a, run_cv_ridge(X, y, a)) for a in alphas), key=lambda r: r[1])
    alpha_opt, cv_rmse = results[0]
    scaler = StandardScaler()
    X_norm = scaler.fit_transform(X)
    model = Ridge(alpha=alpha_opt).fit(X_norm, y)
    importance = pd.DataFrame({
        'Feature': X.columns, 'Coefficient': model.coef_, 'Importance': np.abs(model.coef_)
    }).sort_values('Importance', ascending=False)
    return model, cv_rmse, alpha_opt, scaler, importance

model_tg_ridge, cv_tg_ridge, alpha_tg, scaler_tg_ridge, feats_tg = cv_lr(X_tg, y_tg)
model_egc_ridge, cv_egc_ridge, alpha_egc, scaler_egc_ridge, feats_egc = cv_lr(X_egc, y_egc)
print(f"tg  Ridge (full features) CV RMSE: {cv_tg_ridge:.4f}")
print(f"egc Ridge (full features) CV RMSE: {cv_egc_ridge:.4f}")

top_feats_tg = feats_tg['Feature'].head(20).tolist()
top_feats_egc = feats_egc['Feature'].head(20).tolist()
_, cv_tg_top20, _, _, _ = cv_lr(X_tg[top_feats_tg], y_tg)
_, cv_egc_top20, _, _, _ = cv_lr(X_egc[top_feats_egc], y_egc)
print(f"tg  Ridge (top-20 subset)  CV RMSE: {cv_tg_top20:.4f}  <- worse")
print(f"egc Ridge (top-20 subset)  CV RMSE: {cv_egc_top20:.4f}  <- worse")

tg  Ridge (full features) CV RMSE: 47.9849
egc Ridge (full features) CV RMSE: 0.6434


tg  Ridge (top-20 subset)  CV RMSE: 58.5419  <- worse
egc Ridge (top-20 subset)  CV RMSE: 0.9583  <- worse


**Finding:** the top-20 subset scores *worse* on CV for both targets.
The full (cleaned) feature set is kept for the final model rather than
the smaller subset, since CV evidence — not an assumption that "fewer,
important-looking features must generalize better" — should decide this.

## 5. Phase 2 — HistGradientBoosting

A tree-based model sidesteps Ridge's sensitivity to feature scale and
collinearity entirely. Quick to check since it's built into scikit-learn
(no new dependencies) and fast even on a single CPU core.

In [6]:
def cv_histgb(X, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    errors = []
    for tr, va in kf.split(X):
        model = HistGradientBoostingRegressor(random_state=42)
        model.fit(X.values[tr], y.values[tr])
        errors.append(root_mean_squared_error(y.values[va], model.predict(X.values[va])))
    return np.mean(errors)

cv_tg_hgb = cv_histgb(X_tg, y_tg)
cv_egc_hgb = cv_histgb(X_egc, y_egc)
print(f"tg  HistGB CV RMSE: {cv_tg_hgb:.4f}")
print(f"egc HistGB CV RMSE: {cv_egc_hgb:.4f}")

tg  HistGB CV RMSE: 40.0084
egc HistGB CV RMSE: 0.5510


## 6. Results so far

| Target | Ridge (full) | Ridge (top-20) | HistGB | Winner |
|---|---|---|---|---|
| tg  | ~48.0 | ~58.5 (worse) | **~40.0** | HistGB |
| egc | ~0.64 | ~0.96 (worse) | **~0.55** | HistGB |

(target scale: tg mean≈140, std≈109; egc mean≈4.5, std≈1.56 — so both
models explain a solid majority of variance; HistGB wins on both targets
and is used for the final submission.)

## 7. Final model + submission

In [7]:
model_tg_final = HistGradientBoostingRegressor(random_state=42).fit(X_tg.values, y_tg.values)
model_egc_final = HistGradientBoostingRegressor(random_state=42).fit(X_egc.values, y_egc.values)

# Same cleaning as train, applied to test -- this is the step the original
# baseline skipped (see section 3).
X_tg_test = tg_feat_test[X_tg.columns].replace([np.inf, -np.inf], np.nan)
X_egc_test = egc_feat_test[X_egc.columns].replace([np.inf, -np.inf], np.nan)
n_bad_tg = int(X_tg_test.isnull().sum().sum())
n_bad_egc = int(X_egc_test.isnull().sum().sum())
print(f"Test inf/NaN cells found -> tg: {n_bad_tg}, egc: {n_bad_egc} (filled with train column means)")
X_tg_test = X_tg_test.fillna(X_tg.mean())
X_egc_test = X_egc_test.fillna(X_egc.mean())

tg_pred = model_tg_final.predict(X_tg_test.values)
egc_pred = model_egc_final.predict(X_egc_test.values)

test_tg_out = test_tg.copy(); test_tg_out['target'] = tg_pred
test_egc_out = test_egc.copy(); test_egc_out['target'] = egc_pred
submission = pd.concat([test_tg_out, test_egc_out], axis=0)[['id', 'target']].sort_values('id').reset_index(drop=True)

# Sanity checks before writing the file -- cheap insurance against a
# silently malformed submission
assert len(submission) == len(test), "row count mismatch vs test.csv"
assert submission.isnull().sum().sum() == 0, "NaNs present in final submission"
assert set(submission['id']) == set(test['id']), "id sets don't match test.csv"

submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()

Test inf/NaN cells found -> tg: 10224, egc: 5382 (filled with train column means)
(4115, 2)


,id,target
0,1,300.633405
1,2,4.980760
2,3,50.942859
3,4,54.708924
4,5,72.586162


## Notes & honest limitations

- No hyperparameter tuning was done on HistGB (defaults only) — a natural
  next step if there's time before later phases.
- Feature set is limited to RDKit's default 2D descriptors. Graph-based
  representations (e.g. message-passing GNNs on the polymer graph) or
  learned embeddings (PolyBERT-style) would likely do better but need
  more time/compute than a first submission allows.
- The `egc` target's physical meaning wasn't confirmed against the
  competition's data description (Kaggle page is registration-gated) —
  worth double-checking there.
- CV RMSE is an estimate; the actual leaderboard score depends on the
  hidden test set and the competition's specific scoring metric, which
  may differ from plain RMSE per target (e.g. a weighted multi-target
  metric).